# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [25]:
ПРІЗВИЩЕ_ІМЯ = 'Бугай Софія'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-34'
НОМЕР        = 2       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [26]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 2   ·   Бугай Софія   ·   ШІ-34

Ваші вісім стовпців:
   1. Телефон
   2. Супутній
   3. ЛН
   4. Дата розтину
   5. КТ
   6. Фібрин1
   7. Сечов2
   8. ФЕРИТ1

Ваша пара для аналізу зв’язку: ЛН та помер/жив

Ваше додаткове питання:
   Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.


In [27]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Супутній,ЛН,Дата розтину,КТ,Фібрин1,Сечов2,ФЕРИТ1
0,0958469788 дружина Ганна,"ІХС, КС, СН1, НВ ФК2",ІІІ-IV,2022-03-11,20-30%,4,6.5,NaN
1,"0671778486, 0994887663 дружина Оксана","ЦД, ІІ тип",ІІ-ІІІ,NaT,30-35%,5,4,839.0
2,"0958768045, чоловік Степан 0951784726","ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глю...",ІІ,NaT,70-80%,5.29,10.1,NaN
3,0959819331. дочка Валентина 0960845205,"ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусов...",ІІІ-IV,NaT,0.8,2.5,12.5,NaN
4,"0952294866, дочка Дарина 0678360161",ІХС ДК ГХІІ-ІІІ СНІІа ГЛШст3,ІІ-ІІІ,NaT,NaN,4.29,7.4,NaN
5,0959922983 сестра Софія,"ІХС ДК ГХІІ ГЛШ СНІІ ФК2, ЦД ІІ",ІІ-ІІІ,2021-12-26,30-40%,5,21.4,NaN
6,"0998073702, дружина Наталія 0998382226","ІХС ДК ГХ ІІа ФК3, ЦД вперше виявлений",ІІ-ІІІ,NaT,60-65%,3.24,6.1,NaN
7,"0637258771, син Сергій 0968413301","ІХС ДК ГХІІ ГЛШ СНІІа ФК3, полімедик алергія,ерозивий г/...",І,NaT,до 17%,5.04,7.6,NaN


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [28]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Телефон       120 non-null    object        
 1   Супутній      115 non-null    object        
 2   ЛН            121 non-null    object        
 3   Дата розтину  61 non-null     datetime64[ns]
 4   КТ            44 non-null     object        
 5   Фібрин1       115 non-null    object        
 6   Сечов2        102 non-null    object        
 7   ФЕРИТ1        2 non-null      float64       
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 7.7+ KB


In [29]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Супутній
    непорожніх 115 з 121 | унікальних 91 | типи: str×115
    приклади: ["'ІХС, КС, СН1, НВ ФК2'", "'ЦД, ІІ тип'", "'ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глюкози'", "'ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусова тахік, ГХІІ, СНІІв '"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'7

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [30]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),
    'Телефон': ('ідентифікатор', 'Це не величина, а будь-які дії між ними не несуть важливої інформації'),
    'Супутній': ('номінальна', 'Це вільний текст, знначення яких можна лише порівняти на рівність'),
    'ЛН': ('порядкова', 'Порядок ступенів природний, але відстані між ними не визначені'),
    'Дата розтину': ('інтервальна', 'Різниця між датами має сенс, але нульова дата умовна'),
    'КТ': ('відношень', 'Нуль природний, а відношення між величинами мають сенс'),
    'Фібрин1': ('відношень', 'Лабораторна концентрація -- нуль означає, що речовини немає, а в 2 рази більше означає вдвічі більшу концентрацію'),
    'Сечов2': ('відношень', 'Лабораторна концентрація -- нуль означає, що речовини немає, а в 2 рази більше означає вдвічі більшу концентрацію'),
    'ФЕРИТ1': ('відношень', 'Лабораторна концентрація -- нуль означає, що речовини немає, а в 2 рази більше означає вдвічі більшу концентрацію'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,"Це не величина, а будь-які дії між ними не несуть важлив..."
1,Супутній,номінальна,"Це вільний текст, знначення яких можна лише порівняти на..."
2,ЛН,порядкова,"Порядок ступенів природний, але відстані між ними не виз..."
3,Дата розтину,інтервальна,"Різниця між датами має сенс, але нульова дата умовна"
4,КТ,відношень,"Нуль природний, а відношення між величинами мають сенс"
5,Фібрин1,відношень,"Лабораторна концентрація -- нуль означає, що речовини не..."
6,Сечов2,відношень,"Лабораторна концентрація -- нуль означає, що речовини не..."
7,ФЕРИТ1,відношень,"Лабораторна концентрація -- нуль означає, що речовини не..."


In [31]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [32]:
for v in df['ЛН'].dropna().unique():
    print(repr(v), [hex(ord(ch)) for ch in v])

'ІІІ-IV' ['0x406', '0x406', '0x406', '0x2d', '0x49', '0x56']
'ІІ-ІІІ' ['0x406', '0x406', '0x2d', '0x406', '0x406', '0x406']
'ІІ' ['0x406', '0x406']
'І' ['0x406']
'ІІІ' ['0x406', '0x406', '0x406']
'І-ІІ' ['0x406', '0x2d', '0x406', '0x406']


In [33]:
def нормалізуй_лн(s):
    return (str(s).strip()
            .replace('\u0406', 'I')
            # .replace('\u0456', 'I')
            .replace('–', '-').replace('—', '-')
            .replace(' ', '').upper())

ПОРЯДОК = {'ЛН': ['I', 'I-II', 'II', 'II-III', 'III', 'III-IV']}  # звірте з кроком 1

def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    р['частоти'] = з.value_counts().to_dict()
    if ш in ('ідентифікатор',):
        return р
    р['мода'] = з.mode().tolist()
    if ш == 'номінальна':
        return р

    if ш == 'порядкова':
        cats = ПОРЯДОК[стовпець]
        норм = з.map(нормалізуй_лн)
        невідомі = sorted(set(норм) - set(cats))
        if невідомі:
            р['немає в ПОРЯДОК'] = невідомі
        норм = норм[норм.isin(cats)]
        if норм.empty if False else норм.empty:
            р[''] = 'жодне значення не збіглося з ПОРЯДОК'
            return р
        коди = норм.map({c: i for i, c in enumerate(cats)})
        for q in (0.25, 0.5, 0.75):
            р[f'квантиль {q}'] = cats[int(np.quantile(коди, q, method='lower'))]
        return р

    if pd.api.types.is_datetime64_any_dtype(з):
        чис = з 
    else:
        чис = pd.to_numeric(з, errors='coerce').dropna()
        відкинуто = len(з) - len(чис)
        if відкинуто:
            р['не-числових відкинуто'] = відкинуто

    р['медіана'] = чис.median()
    р['q25'], р['q75'] = чис.quantile(0.25), чис.quantile(0.75)
    р['середнє'] = чис.mean()
    р['ст. відхилення'] = чис.std()
    if ш == 'інтервальна':
        return р

    р['коеф. варіації'] = чис.std() / чис.mean()
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120, 'частоти': {'0958469788 дружина Ганна': 1, '0671778486, 0994887663 дружина Оксана': 1, '0958768045, чоловік Степан 0951784726': 1, '0959819331. дочка Валентина 0960845205': 1, '0952294866, дочка Дарина 0678360161': 1, '0959922983 сестра Софія': 1, '0998073702, дружина Наталія 0998382226': 1, '0637258771, син Сергій 0968413301': 1, '0963194893, сестра Катерина 0964948214': 1, '0635244106 син Микола': 1, 504118955: 1, 955939761: 1, 635060454: 1, '0679110285 подруга Зоряна': 1, '0993228088, син Степан 0954230048': 1, 961280845: 1, 990370382: 1, 'син владислав 0997879432': 1, 'син Ярослав 0638421083': 1, '0638440932, дружина Ганна 0634845263': 1, 688280258: 1, 'син Володимир 0679932157': 1, '0675795111, дочка 0954432568': 1, '0966327926, син 0635154742': 1, 'дружина Юлія 0950856517': 1, '0675996775, син Сергій 0500314503': 1, '0953924196, чол.Володимир 0998982104': 1, '0954754072, дочка Христина 0507547354': 1, 5061884

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [34]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Супутній,115,91,ІХС ДК ГХст2,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049088,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Фібрин1,115.0,87.0,6.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Сечов2,102.0,66.0,4.9,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ФЕРИТ1,2.0,NaN,NaN,NaN,555.05,271.1,413.075,555.05,697.025,839.0,401.565941


In [35]:
БЕЗГЛУЗДІ = [
    # ('стовпець', 'яка статистика', значення, 'чому це число нічого не означає'),
    ('Телефон', 'mean, min, max, std, проміжні', np.nan , 'Номер телефону є набором цифр, не числом'),
    ('ЛН', 'mean, min, max, std, проміжні', np.nan , 'Хоча і можна вважати числом (перша, друга і тд. стадії), однак дії між ними окрім упорядкування не мають сенсу'),
    ('Дата розтину', 'mean', '2022-04-06 13:22:37.377049088' , 'Середнє від дати не надає жодної інформації про розтини, або навіть про те, коли вони проводились'),
    ('КТ', 'mean, min, max, std, проміжні', np.nan , 'Хоча це і є відсотки, вони не унормовані, тобто дії над їхніми значеннями не дадуть правильної відповіді (деякі зазначені проміжками, деякі тестом зі знаком %, деякі як 80%, а деякі як 0.8)'),
    ('ФЕРИТ1', 'std', 401.565941, 'Лише 2 спостереження не оцінює розкид популяції.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,"mean, min, max, std, проміжні",NaN,"Номер телефону є набором цифр, не числом"
1,ЛН,"mean, min, max, std, проміжні",NaN,"Хоча і можна вважати числом (перша, друга і тд. стадії),..."
2,Дата розтину,mean,2022-04-06 13:22:37.377049088,"Середнє від дати не надає жодної інформації про розтини,..."
3,КТ,"mean, min, max, std, проміжні",NaN,"Хоча це і є відсотки, вони не унормовані, тобто дії над ..."
4,ФЕРИТ1,std,401.565941,Лише 2 спостереження не оцінює розкид популяції.


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [36]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: ЛН × помер/жив

ЛН: шкала — порядкова
помер/жив: шкала — не у вашому списку

       ЛН  помер/жив
0  ІІІ-IV          1
1  ІІ-ІІІ          2
2      ІІ          2
3  ІІІ-IV          2
4  ІІ-ІІІ          2
5  ІІ-ІІІ          1


In [37]:
шкала_A = 'номінальна'
ОБРАНА_МІРА = 'коефіцієнт Крамера (V)'        # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = 'ЛН за шкалою є порядковою, а помер/живий - номінальною (тільки порівняти з наданими значеннями), тому за таблицею було обрано коефіцієнт Крамера (V). Даний коефіцієнт використовується для вимірювання зв\'язку між двома категоріальними змінними.'     # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# ваш код: обчисліть обрану міру
from scipy.stats import chi2_contingency

таб = pd.crosstab(df[A].map(нормалізуй_лн), df[B])  
print(таб)

хі2, p, _, очік = chi2_contingency(таб, correction=False)
n = таб.values.sum()
V = np.sqrt(хі2 / (n * (min(таб.shape) - 1)))

print(f'χ² = {хі2:.2f}, p = {p:.4f}, n = {n}')
print(f'V Крамера = {V:.3f}')
print(f'найменша очікувана частота: {очік.min():.2f}')

помер/жив   1   2
ЛН               
I           0  12
I-II        0   9
II          0  21
II-III     49  10
III        11   6
III-IV      1   2
χ² = 69.58, p = 0.0000, n = 121
V Крамера = 0.758
найменша очікувана частота: 1.49


V Крамера = 0,758. Це сильний зв'язок. Не можна ствердити, що смертність залежить саме від ступеню ЛН, однак випадковістю це теж не є.

In [38]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
ранги = {'I': 1, 'I-II': 2, 'II': 3, 'II-III': 4, 'III': 5, 'III-IV': 6}
x = df[A].map(нормалізуй_лн).map(ранги)
y = df[B]

ok = x.notna() & y.notna()
x, y = x[ok], y[ok]

r1 = np.corrcoef(x, y)[0, 1]
r2 = np.corrcoef(x, 3 - y)[0, 1]
r3 = np.corrcoef(x.map(lambda k: k**2), y)[0, 1]

print(f'Пірсон, початкове кодування:        {r1:+.3f}')
print(f'Пірсон, мітки помер/жив поміняли:   {r2:+.3f}')
print(f'Пірсон, ранги ЛН піднесли до квадр.: {r3:+.3f}')

Пірсон, початкове кодування:        -0.544
Пірсон, мітки помер/жив поміняли:   +0.544
Пірсон, ранги ЛН піднесли до квадр.: -0.495


Пірсон дає |r| ≈ 0,5, тобто «середній зв'язок», тоді як V Крамера дає 0,758, тобто сильний. Пірсон занижує зв'язок, бо шукає пряму лінію, а реальна залежність має форму сходинки з немонотонним хвостом. Це додатковий аргумент проти Пірсона: він не тільки довільний за знаком, а й неадекватний за формою залежності.

---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [39]:
ПІДГОТОВКА = {
    'Телефон':      'не беремо в модель: ідентифікатор, не величина',
    'Супутній':     'групуємо за ключовими словами, потім one-hot: порядок категорій не вводимо',
    'ЛН':           'ціле кодування 1..6 із заданим порядком: порядок відомий, відстані ні',
    'Дата розтину': 'перехід до тривалості в днях від найранішої дати, потім стандартизація',
    'КТ':           'розпарсити в частку 0..1, потім нормалізація',
    'Фібрин1':      'перетворити на число, log1p проти правого хвоста, потім стандартизація',
    'Сечов2':       'перетворити на число, log1p, стандартизація',
    'ФЕРИТ1':       'не беремо в модель: лише 2 значення',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,"не беремо в модель: ідентифікатор, не величина"
1,Супутній,номінальна,"групуємо за ключовими словами, потім one-hot: порядок ка..."
2,ЛН,порядкова,ціле кодування 1..6 із заданим порядком: порядок відомий...
3,Дата розтину,інтервальна,"перехід до тривалості в днях від найранішої дати, потім ..."
4,КТ,відношень,"розпарсити в частку 0..1, потім нормалізація"
5,Фібрин1,відношень,"перетворити на число, log1p проти правого хвоста, потім ..."
6,Сечов2,відношень,"перетворити на число, log1p, стандартизація"
7,ФЕРИТ1,відношень,не беремо в модель: лише 2 значення


In [40]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
from sklearn.preprocessing import StandardScaler

порядок = ['I', 'I-II', 'II', 'II-III', 'III', 'III-IV']

лн = df['ЛН'].map(нормалізуй_лн)
лн_кат = pd.Categorical(лн, categories=порядок, ordered=True)
лн_код = pd.Series(лн_кат.codes, index=df.index).replace(-1, np.nan) + 1
print('ЛН невідомих:', int(лн_код.isna().sum()))
print(лн_код.value_counts().sort_index())

def група(t):
    if pd.isna(t): return 'невідомо'
    t = str(t).upper().replace('І', 'I')
    if 'ІХС' in t or 'IXC' in t or 'ИБС' in t: return 'ІХС'
    if 'ЦД' in t or 'ДІАБЕТ' in t or 'DM' in t: return 'ЦД'
    if 'ГХ' in t or 'ГІПЕРТОН' in t:            return 'ГХ'
    return 'інше'

супутні = df['Супутній'].map(група)
супутні_oh = pd.get_dummies(супутні, prefix='супутній')
print(супутні.value_counts())
print(супутні_oh.head())

фібрин = pd.to_numeric(df['Фібрин1'], errors='coerce')
print('Фібрин1 не розпізнано:', int(fібрин.isna().sum() - df['Фібрин1'].isna().sum()) if False else int(фібрин.isna().sum() - df['Фібрин1'].isna().sum()))
ф_log = np.log1p(фібрин)
ok = ф_log.notna()
ф_z = pd.Series(np.nan, index=df.index)
ф_z[ok] = StandardScaler().fit_transform(ф_log[ok].to_frame()).ravel()
print(f'до: mean={фібрин.mean():.2f}, std={фібрин.std():.2f}')
print(f'після: mean={ф_z.mean():.3f}, std={ф_z.std():.3f}')

ЛН невідомих: 0
1    12
2     9
3    21
4    59
5    17
6     3
Name: count, dtype: int64
Супутній
ГХ          50
інше        37
ЦД          28
невідомо     6
Name: count, dtype: int64
   супутній_ГХ  супутній_ЦД  супутній_невідомо  супутній_інше
0        False        False              False           True
1        False         True              False          False
2         True        False              False          False
3        False         True              False          False
4         True        False              False          False
Фібрин1 не розпізнано: 1
до: mean=5.05, std=1.95
після: mean=0.000, std=1.004


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [41]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [42]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [43]:
ВИПРАВЛЕННЯ = [
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
    ('1. Результат лікування',
     'Середнє від номінальної бінарної ознаки (коди 1 і 2 є лише мітками).',
     'Число 1,50 залежить від довільних кодів: з мітками 0/1 було б 0,50. «Рівно посередині між помер і вижив» не є станом жодного пацієнта.',
     'Рахувати частоти й частки: 61 з кодом 1 (50,4 %) і 60 з кодом 2 (49,6 %). Середнє лише як частка події після кодування 0/1.'),

    ('2. Стать',
     'Дані непридатні: одна номінальна ознака записана чотирма написаннями (ж, жін, ч, чол).',
     '«Чотири статі» є артефактом введення. Насправді дві категорії, а розподіл 56/34/19/12 розбитий на штучні групи й дає хибний висновок.',
     'Привести до єдиного запису (ж+жін = 75, ч+чол = 46) і лише потім рахувати частоти. Спершу очищення категорій, потім статистика.'),

    ('3. Ступінь КТ',
     'Порядкова шкала: середнє від кодів, які присвоєно за алфавітом на брудних мітках («важкий» двічі, «сердній», «серед ступ» тощо).',
     'sort_index() упорядковує літери, а не тяжкість: «найлегшим» став «важкий». Середнє 5,9 → «легкий» не має змісту: відстані між ступенями невідомі.',
     'Нормалізувати мітки, задати порядок вручну (легкий < середній < важкий < вкрай важкий < критичний) і брати лише медіану та квантилі.'),

    ('4. Обсяг ураження КТ',
     'Дані непридатні: to_numeric(errors="coerce") мовчки відкинув рядкові значення («20-30%») і працює на частині записів.',
     'Із 44 заповнених значень 17 рядкових, тож 39,6 % отримано приблизно на 27 пацієнтах із 121, до того ж несистематично відібраних.',
     'Розпарсити діапазони (середина інтервалу), перевести все в єдині одиниці і завжди друкувати n. Або явно написати, що показник заповнено лише у частини.'),

    ('5. Сатурація',
     'Дані непридатні: у стовпці є мінімум 0,9 %, що фізіологічно неможливо (помилка введення чи інша одиниця).',
     'Викид тягне середнє донизу (82,05 проти медіани 84,0), а висновок про «більшість пацієнтів» не випливає зі середнього.',
     'Знайти й виправити/виключити неможливі значення, порахувати медіану та квартилі і частку пацієнтів нижче порога (наприклад, < 90 %).'),

    ('6. Дати госпіталізації',
     'Коефіцієнт варіації для дат: інтервальна шкала має умовний нуль, а CV вимагає природного нуля.',
     'CV = σ/середнє залежить від точки відліку. Для тих самих дат він змінюється від 0,3 % (відлік від 1970) до десятків відсотків (від 2021). Про «однорідність потоку» 0,6 % нічого не каже.',
     'Для дат допустимі різниці: σ у днях, розмах, кількість надходжень по тижнях/місяцях. CV відкинути.'),

    ('7. Стать і результат',
     'Номінальні ознаки: «лінійний зв’язок» і знак кореляції не мають змісту, а 0,177 не є «близьким до нуля».',
     'Для двох бінарних ознак r дорівнює φ. Тоді χ² = n·φ² ≈ 3,8 і p ≈ 0,05: зв’язок межовий, «не виявлено» є надто сильним твердженням. Знак залежить від довільних кодів.',
     'Таблиця спряженості, χ² або Фішер, φ/V Крамера з довірчим інтервалом і чесним висновком: «не можна ні підтвердити, ні виключити».'),

    ('8. Нормалізація',
     'Вибір ознак за dtype (select_dtypes("number")), а не за шкалою: до мін-макс потрапили ідентифікатори, бінарні коди, цільова змінна.',
     'Масштабування лише для величин (інтервальна, відношень). Для кодів категорій воно безглузде, ФЕРИТ2 порожній, а викиди (як 0,9) стискають решту значень.',
     'Відібрати стовпці за паспортом шкал, пропустити номінальні й ідентифікатори, прибрати порожні й підганяти скейлер лише на тренувальній вибірці.'),

    ('9. Очищення пропусків',
     'Дані непридатні: dropna() видаляє рядок при будь-якому пропуску, а в 77 ознаках є порожні й майже порожні стовпці.',
     'Результат 0 рядків, але текст жорстко пише «повністю заповнений, придатний для моделі». Висновок суперечить власному виводу.',
     'Спершу оцінити частку пропусків по стовпцях, прибрати порожні/майже порожні, а для решти обґрунтувати імпутацію. Потім перевірити n.'),

    ('10. Рейтинг ознак',
     'Витік цілі: «Дата розтину» існує лише в тих, хто помер, а NaN після astype(int64) стає −9,2·10^18.',
     'r = 1,00 відображає не прогноз, а те, що розтин проводять після смерті (61 дата = 61 пацієнт із кодом 1). Решта r на різних n, тож несумірні.',
     'Виключити всі ознаки, відомі лише після результату, і вказувати n для кожної ознаки. Для ранжування брати рангові міри й порівнювати лише при співмірних n.'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1. Результат лікування,Середнє від номінальної бінарної ознаки (коди 1 і 2 є ли...,"Число 1,50 залежить від довільних кодів: з мітками 0/1 б...","Рахувати частоти й частки: 61 з кодом 1 (50,4 %) і 60 з ..."
1,2. Стать,Дані непридатні: одна номінальна ознака записана чотирма...,«Чотири статі» є артефактом введення. Насправді дві кате...,"Привести до єдиного запису (ж+жін = 75, ч+чол = 46) і ли..."
2,3. Ступінь КТ,"Порядкова шкала: середнє від кодів, які присвоєно за алф...","sort_index() упорядковує літери, а не тяжкість: «найлегш...","Нормалізувати мітки, задати порядок вручну (легкий < сер..."
3,4. Обсяг ураження КТ,"Дані непридатні: to_numeric(errors=""coerce"") мовчки відк...","Із 44 заповнених значень 17 рядкових, тож 39,6 % отриман...","Розпарсити діапазони (середина інтервалу), перевести все..."
4,5. Сатурація,"Дані непридатні: у стовпці є мінімум 0,9 %, що фізіологі...","Викид тягне середнє донизу (82,05 проти медіани 84,0), а...","Знайти й виправити/виключити неможливі значення, порахув..."
5,6. Дати госпіталізації,Коефіцієнт варіації для дат: інтервальна шкала має умовн...,CV = σ/середнє залежить від точки відліку. Для тих самих...,"Для дат допустимі різниці: σ у днях, розмах, кількість н..."
6,7. Стать і результат,Номінальні ознаки: «лінійний зв’язок» і знак кореляції н...,Для двох бінарних ознак r дорівнює φ. Тоді χ² = n·φ² ≈ 3...,"Таблиця спряженості, χ² або Фішер, φ/V Крамера з довірчи..."
7,8. Нормалізація,"Вибір ознак за dtype (select_dtypes(""number"")), а не за ...","Масштабування лише для величин (інтервальна, відношень)....","Відібрати стовпці за паспортом шкал, пропустити номіналь..."
8,9. Очищення пропусків,Дані непридатні: dropna() видаляє рядок при будь-якому п...,"Результат 0 рядків, але текст жорстко пише «повністю зап...","Спершу оцінити частку пропусків по стовпцях, прибрати по..."
9,10. Рейтинг ознак,"Витік цілі: «Дата розтину» існує лише в тих, хто помер, ...","r = 1,00 відображає не прогноз, а те, що розтин проводят...","Виключити всі ознаки, відомі лише після результату, і вк..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [44]:
# ваш код: три правильні обчислення замість хибних
print('Розділ 1')
print('ХИБНО:', df['помер/жив'].mean())
print('ПРАВИЛЬНО:')
print(df['помер/жив'].value_counts())
print((df['помер/жив'].value_counts(normalize=True) * 100).round(1))
print('-'*35, '\n')

print('Розділ 3')
порядок = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
мапа = {'легка': 'легкий', 'сердній': 'середній',
        'серед ступ': 'середній', 'серед ступеня': 'середній'}

мітки = (df['ступінь КТ'].dropna().astype(str)
         .str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
         .replace(мапа))
print('Невідомі мітки:', sorted(set(мітки) - set(порядок)))

коди = pd.Categorical(мітки, categories=порядок, ordered=True).codes
коди = pd.Series(коди)[коди >= 0]
for q in (0.25, 0.5, 0.75):
    print(f'квантиль {q}:', порядок[int(np.quantile(коди, q, method="lower"))])
print(pd.Series(pd.Categorical(мітки, categories=порядок, ordered=True)).value_counts(sort=False))
print('-'*35, '\n')

print('Розділ 6')
дати = pd.to_datetime(df['Поступлення'], errors='coerce').dropna()
for відлік in ['1970-01-01', '2021-01-01']:     
    x = (дати - pd.Timestamp(відлік)).dt.days
    print(f'ХИБНО: CV від {відлік} = {100*x.std()/x.mean():.2f} %')
print('ПРАВИЛЬНО:')
print('σ у днях :', round((дати - дати.min()).dt.days.std(), 1))
print('розмах   :', дати.min().date(), '—', дати.max().date())
print(дати.dt.to_period('M').value_counts().sort_index())

print('ХИБНО: NaT ->', pd.Series(pd.to_datetime([None])).astype('int64')[0])
print('Є дата розтину:', df['Дата розтину'].notna().sum(),
      '| код 1 (помер):', (df['помер/жив'] == 1).sum())
print(pd.crosstab(df['Дата розтину'].notna(), df['помер/жив']))
print('-'*35, '\n')

Розділ 1
ХИБНО: 1.4958677685950412
ПРАВИЛЬНО:
помер/жив
1    61
2    60
Name: count, dtype: int64
помер/жив
1    50.4
2    49.6
Name: proportion, dtype: float64
----------------------------------- 

Розділ 3
Невідомі мітки: []
квантиль 0.25: середній
квантиль 0.5: середній
квантиль 0.75: важкий
легкий           7
середній        18
важкий          10
вкрай важкий     4
критичний        3
Name: count, dtype: int64
----------------------------------- 

Розділ 6
ХИБНО: CV від 1970-01-01 = 0.59 %
ХИБНО: CV від 2021-01-01 = 30.04 %
ПРАВИЛЬНО:
σ у днях : 113.0
розмах   : 2021-02-25 — 2022-06-03
Поступлення
2021-02     1
2021-04     2
2021-05     4
2021-06     3
2021-07     2
2021-08     4
2021-09     9
2021-10     4
2021-11    11
2021-12     7
2022-01     3
2022-02    10
2022-03    18
2022-04    30
2022-05    11
2022-06     1
Freq: M, Name: count, dtype: int64
ХИБНО: NaT -> -9223372036854775808
Є дата розтину: 61 | код 1 (помер): 61
помер/жив      1   2
Дата розтину        
False          0 

---
# Етап 7. Висновки та індивідуальне питання

In [45]:
ВИСНОВКИ = [
    'Серед восьми стовпців лише Фібрин1, Сечов2, ФЕРИТ1 і КТ є величинами зі шкалою відношень, а Телефон є ідентифікатором, Супутній номінальним, ЛН порядковим, Дата розтину інтервальним, тому до них не можна застосовувати однакові статистики лише на підставі того, що pandas читає частину з них як числа',

    'Якість даних обмежує аналіз: у КТ заповнено лише 44 значення зі 121 і до того ж у змішаному форматі (27 чисел і 17 діапазонів на кшталт «20-30%»), у ФЕРИТ1 лише 2 значення, а у Фібрин1 та Сечов2 трапляється по одному рядковому значенню серед чисел, тож їхні середні й відхилення без попереднього очищення ненадійні',

    'Між ступенем ЛН і результатом лікування виявлено сильний зв’язок (χ² = 69,6; V Крамера = 0,76; n = 121): усі 42 пацієнти зі ступенями I, I-II та II мають код 2, а при II-III код 1 мають 49 із 59 (83 %), однак мінімальна очікувана частота становить лише 1,49, тому точне значення V слід вважати орієнтовним',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Серед восьми стовпців лише Фібрин1, Сечов2, ФЕРИТ1 і КТ є величинами зі шкалою відношень, а Телефон є ідентифікатором, Супутній номінальним, ЛН порядковим, Дата розтину інтервальним, тому до них не можна застосовувати однакові статистики лише на підставі того, що pandas читає частину з них як числа
2. Якість даних обмежує аналіз: у КТ заповнено лише 44 значення зі 121 і до того ж у змішаному форматі (27 чисел і 17 діапазонів на кшталт «20-30%»), у ФЕРИТ1 лише 2 значення, а у Фібрин1 та Сечов2 трапляється по одному рядковому значенню серед чисел, тож їхні середні й відхилення без попереднього очищення ненадійні
3. Між ступенем ЛН і результатом лікування виявлено сильний зв’язок (χ² = 69,6; V Крамера = 0,76; n = 121): усі 42 пацієнти зі ступенями I, I-II та II мають код 2, а при II-III код 1 мають 49 із 59 (83 %), однак мінімальна очікувана частота становить лише 1,49, тому точне значення V слід вважати орієнтовним


In [46]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = 'Є такі три способи перетворити його на числові ознаки: 1. Розділити на два числа: систолічний (САТ) і діастолічний (ДАТ). Це найпряміший спосіб: «140/100» стає двома ознаками, 140 і 100. Обидві величини вимірюються в мм рт. ст. зі шкалою відношень, нуль означає відсутність тиску. Нічого не втрачається, а інтерпретація зрозуміла лікарю. 2. Похідні показники: пульсовий тиск і середній артеріальний тиск. Пульсовий тиск (ПТ) дорівнює САТ − ДАТ, а середній (СрАТ) дорівнює ДАТ + (САТ − ДАТ)/3. Обидва мають фізіологічний зміст: ПТ відображає жорсткість судин і ударний обсяг, СрАТ відображає перфузію органів. Але вони обчислюються з тих самих двох чисел, тож нової інформації не додають, лише інакше її впаковують. 3. Клінічні категорії. Наприклад: гіпотензія, норма, гіпертензія 1, 2, 3 ступеня за порогами. Це порядкова ознака, її кодують цілими числами із заданим порядком. Шкала знижується з відношень до порядкової, тому частина інформації втрачається.\nУ модель моїм вибором було би взяти перший варіант - дані не втрачаються, інші значення легко обраховуються з них самих.'
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.

Є такі три способи перетворити його на числові ознаки: 1. Розділити на два числа: систолічний (САТ) і діастолічний (ДАТ). Це найпряміший спосіб: «140/100» стає двома ознаками, 140 і 100. Обидві величини вимірюються в мм рт. ст. зі шкалою відношень, нуль означає відсутність тиску. Нічого не втрачається, а інтерпретація зрозуміла лікарю. 2. Похідні показники: пульсовий тиск і середній артеріальний тиск. Пульсовий тиск (ПТ) дорівнює САТ − ДАТ, а середній (СрАТ) дорівнює ДАТ + (САТ − ДАТ)/3. Обидва мають фізіологічний зміст: ПТ відображає жорсткість судин і ударний обсяг, СрАТ відображає перфузію органів. Але вони обчислюються з тих самих двох чисел, тож нової інформації не додають, лише інакше її впаковують. 3. Клінічні категорії. Наприклад: гіпотензія, норма, гіпертензія 1, 2, 3 ступеня за порогами. Це порядкова

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [47]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, як є',
    'етап 2 — допустимі статистики':   'текст, як є',
    'етап 3 — безглузді числа':        'текст, як є',
    'етап 4 — міра зв’язку':           'текст, як є',
    'етап 5 — підготовка до моделі':   'текст, як є',
    'етап 6 — виправлення аналізу':    'текст, як є',
    'етап 7 — висновки й відповідь':   'текст, як є',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Увесь код було переглянуто, а результато перевірено самотужки та детально розібрано. Усі типи шкал та міри були обрані самостійно та перевірені'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [48]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 2 готова до захисту.
Бугай Софія, ШІ-34


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.